# TabPFN runs on Colab (GPU)

The three classic models run fine on a normal computer. Only **TabPFN** needs a GPU
(graphics card that speeds up neural-network maths), so this notebook runs TabPFN only.
Results are saved to `results/tables/raw_results_tabpfn.csv` and downloaded to your computer.

**Before starting (one time):** in the 🔑 *Secrets* panel on the left, add `HF_TOKEN`
(Hugging Face read token) and `TABPFN_TOKEN` (Prior Labs API key), and switch on
"Notebook access" for both. Keys are read from Secrets, never typed here.

**Run the cells top to bottom, one at a time.**
- Cell A restarts Python on purpose at the end ("session crashed" message = expected).
- Every cell after A starts with `%cd /content/project`, so the order after a restart is safe.
- "Restart session" keeps your files. "Disconnect and delete runtime" wipes everything.

## 0. GPU check
Runtime → Change runtime type → **T4 GPU** → Save. This should print `Tesla T4`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## A. Get the code and install (once per fresh runtime)
The red "dependency conflicts" lines are about Colab's own extras; they are harmless.

In [ ]:
%cd /content
!rm -rf project
!git clone https://github.com/manvi-xpert1/small-tabular-benchmark.git project
%cd /content/project
!pip install -q -r requirements.txt
import os
os.kill(os.getpid(), 9)   # restart Python so the newly installed packages load

## B. After the restart: keys and version check
Should print `3.1.0 2.5.4 2.14.1... True`.

In [ ]:
%cd /content/project
import os
from google.colab import userdata, files
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # Hugging Face: download TabPFN's files
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")  # Prior Labs: licence check
os.environ["TABPFN_NO_BROWSER"] = "1"                      # skip browser login (can't work in Colab)
import pandas, numpy, torch
print(pandas.__version__, numpy.__version__, torch.__version__, "GPU:", torch.cuda.is_available())
print("Keys loaded:", bool(os.environ["HF_TOKEN"]), bool(os.environ["TABPFN_TOKEN"]))

## C. Upload files
Select these 4 data files from your laptop's `data` folder:
`winequality-red.csv`, `winequality-white.csv`, `adult.data`, `adult.test`.

**Resuming an earlier run?** Also select your downloaded `raw_results_tabpfn.csv`.
Finished runs are then skipped.

In [ ]:
%cd /content/project
import os, shutil
from google.colab import files
os.makedirs("data", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
for name in files.upload():
    if name.endswith(".csv") and name.startswith("raw_results"):
        shutil.move(name, "results/tables/raw_results_tabpfn.csv")   # earlier results, to resume
    else:
        shutil.move(name, f"data/{name}")
print("data:", sorted(os.listdir("data")))
print("results:", os.listdir("results/tables"))

## D. Quick TabPFN check (about 1-3 minutes the first time: it downloads the 876 MB model)

In [ ]:
%cd /content/project
import os
from google.colab import userdata, files
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # Hugging Face: download TabPFN's files
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")  # Prior Labs: licence check
os.environ["TABPFN_NO_BROWSER"] = "1"                      # skip browser login (can't work in Colab)
import time, torch
from src.data import load_dataset
from src.models import make_model
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn the GPU on")
start = time.time()
ds = load_dataset("wine")
m = make_model("tabpfn", ds.numeric, ds.categorical, seed=0)
m.fit(ds.X[:100], ds.y[:100])
print(m.predict_proba(ds.X[100:2100])[:3].round(3))
print(f"Took {time.time() - start:.1f} seconds")

## E. Run TabPFN on Wine, then download
Each line printed = one finished step. The file downloads automatically at the end.
If Colab disconnects: redo A → B → C (upload the downloaded CSV too), then rerun this cell.

**Short on time?** Add `--repeats 10 --sizes 50 100 250 500 --noise-levels 0 0.25` to the
`!python` line. That is about 1/6 of the work, and its numbers match the full run exactly.

In [ ]:
%cd /content/project
import os
from google.colab import userdata, files
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # Hugging Face: download TabPFN's files
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")  # Prior Labs: licence check
os.environ["TABPFN_NO_BROWSER"] = "1"                      # skip browser login (can't work in Colab)
!python -m src.experiment --datasets wine --models tabpfn --out results/tables/raw_results_tabpfn.csv
files.download("results/tables/raw_results_tabpfn.csv")

## F. Run TabPFN on Adult, then download (same file, now containing both datasets)

In [ ]:
%cd /content/project
import os
from google.colab import userdata, files
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # Hugging Face: download TabPFN's files
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")  # Prior Labs: licence check
os.environ["TABPFN_NO_BROWSER"] = "1"                      # skip browser login (can't work in Colab)
!python -m src.experiment --datasets adult --models tabpfn --out results/tables/raw_results_tabpfn.csv
files.download("results/tables/raw_results_tabpfn.csv")

## Done
Save the downloaded `raw_results_tabpfn.csv` into your laptop's
`project 1\results\tables\` folder. The classic-model results and the analysis
are produced on a normal computer (`python -m src.experiment --models logistic_regression random_forest gradient_boosting`).